# MNIST分类：比较不同激活函数与优化器

本项目旨在使用PyTorch框架，在MNIST数据集上训练一个多层感知机（MLP）模型，并系统地比较三种激活函数（ReLU, Sigmoid, Tanh）与三种优化器（SGD, Momentum, Adam）的组合性能。

**实验流程：**
1.  **环境设置**：导入所需库，并自动检测配置GPU（如您的4060）进行训练。
2.  **数据加载与预处理**：加载MNIST数据集，进行标准化处理，并创建PyTorch的`DataLoader`以便高效分批训练。
3.  **模型定义**：构建一个含单隐藏层的MLP模型。
4.  **训练与评估**：遍历所有激活函数与优化器的组合，进行模型训练，并在验证集上评估准确率。
5.  **结果分析**：找出表现最佳的组合，并保存结果。

## 1. 环境设置

导入所有必要的库，包括`torch`、`numpy`以及用于显示进度条的`tqdm`。同时，自动检测并设置CUDA设备（GPU）。

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
import os
import json
from re import search
from tqdm.notebook import tqdm

# 检查是否有可用的CUDA设备 (NVIDIA GPU)，并设置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# 设置随机种子以保证结果可复现
np.random.seed(999)
torch.manual_seed(999)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(999)

Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. 数据加载与预处理

加载本地的`mnist.npz`文件，选取前20,000个样本，进行标准化和训练/验证集划分，最后转换为PyTorch `DataLoader`。

In [6]:
def load_data(path="datasets/mnist.npz"):
    """从指定路径加载MNIST数据集"""
    print(f"Attempting to load data from: {path}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"[ERROR] Data file not found at path: {path}. Please ensure it is downloaded.")
    with np.load(path) as f:
        X_train, y_train = f['x_train'], f['y_train']
        X_test, y_test = f['x_test'], f['y_test']
    return (X_train, y_train), (X_test, y_test)

# 加载数据
(X_train_np, y_train_np), _ = load_data("datasets/mnist.npz")

# 使用前20000个样本
X_train_np, y_train_np = X_train_np[:20000], y_train_np[:20000]

# 展平并归一化图像数据
X_train_np = X_train_np.reshape(X_train_np.shape[0], -1) / 255.0

# 划分训练集和验证集
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_train_np, y_train_np, test_size=0.2, random_state=42
)

# 将Numpy数组转换为PyTorch张量
# 注意：PyTorch的CrossEntropyLoss不需要one-hot编码的标签
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
X_val = torch.tensor(X_val_np, dtype=torch.float32)
y_val = torch.tensor(y_val_np, dtype=torch.long)

# 创建TensorDataset和DataLoader
BATCH_SIZE = 256
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Data loaded and prepared successfully.")

Attempting to load data from: datasets/mnist.npz
Data loaded and prepared successfully.


## 3. 定义MLP模型

使用`torch.nn.Module`构建一个标准的多层感知机。模型结构为：输入层 -> 线性层 -> 激活函数 -> 线性层 -> 输出层。

In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, output_dim=10, activation='relu'):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        
        # 根据字符串选择激活函数
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'sigmoid':
            self.activation = nn.Sigmoid()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x

## 4. 训练与评估循环

这是实验的核心部分。我们将遍历所有激活函数和优化器的组合。

- **外层循环**：遍历激活函数和优化器。
- **内层循环**：进行多个周期的训练。在每个周期中，使用`tqdm`库为数据加载器`train_loader`添加一个进度条，以实时监控训练进度。
- **评估**：每个组合训练完毕后，在验证集上进行评估并打印准确率。

In [9]:
best_accuracy = 0
best_activation = None
best_optimizer = None
EPOCHS = 20 # 使用GPU和PyTorch，20个周期足以获得良好结果

# 定义优化器字典
optimizers_map = {
    'sgd': optim.SGD,
    'momentum': optim.SGD, # Momentum是SGD的一种特殊形式
    'adam': optim.Adam
}

# 损失函数
criterion = nn.CrossEntropyLoss()

for activation_name in ['relu', 'sigmoid', 'tanh']:
    for optimizer_name in ['sgd', 'momentum', 'adam']:
        print(f"\n--- Training with Activation: {activation_name.upper()} & Optimizer: {optimizer_name.upper()} ---")
        
        # 实例化模型并移动到GPU
        model = MLP(activation=activation_name).to(device)
        
        # 实例化优化器
        if optimizer_name == 'momentum':
            optimizer = optimizers_map[optimizer_name](model.parameters(), lr=0.01, momentum=0.9)
        else:
            optimizer = optimizers_map[optimizer_name](model.parameters(), lr=0.001)

        # 训练过程
        for epoch in range(EPOCHS):
            model.train() # 设置为训练模式
            # 使用tqdm添加进度条
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
            for data, targets in progress_bar:
                # 将数据移动到GPU
                data = data.to(device)
                targets = targets.to(device)
                
                # 前向传播
                scores = model(data)
                loss = criterion(scores, targets)
                
                # 反向传播和优化
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                # 更新进度条的描述信息
                progress_bar.set_postfix(loss=loss.item())

        # 评估过程
        model.eval() # 设置为评估模式
        num_correct = 0
        num_samples = 0
        with torch.no_grad(): # 不计算梯度
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)
                
                scores = model(x)
                _, predictions = scores.max(1)
                num_correct += (predictions == y).sum()
                num_samples += predictions.size(0)
        
        accuracy = float(num_correct) / float(num_samples)
        print(f"Validation Accuracy: {accuracy * 100:.2f}%")
        
        # 保存最佳模型信息
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_activation = activation_name
            best_optimizer = optimizer_name


--- Training with Activation: RELU & Optimizer: SGD ---


Epoch 1/20:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 2/20:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 3/20:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 4/20:   0%|          | 0/63 [00:00<?, ?it/s]

...

Epoch 17/20:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 18/20:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 19/20:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 20/20:   0%|          | 0/63 [00:00<?, ?it/s]

Validation Accuracy: 95.93%


## 5. 输出并保存最终结果

打印在所有组合中表现最佳的激活函数和优化器，以及它们在验证集上达到的准确率，并按题目要求将结果保存为`answer_2.json`。

In [10]:
print(f"\n--- Best Model Found ---")
print(f"Activation: '{best_activation}'")
print(f"Optimizer: '{best_optimizer}'")
print(f"Validation Accuracy: {best_accuracy * 100:.2f}%")

# 按题目要求将最优组合赋值给 a1
a1 = [best_activation, best_optimizer]

# 保存为JSON文件
answer = {"q1": a1}

def to_json(answer: dict, file_name: str):
    if not search(r'answer_\d\.json', file_name):
        raise Exception('文件名称格式不符')
    with open(file_name, 'w') as f:
        json.dump(answer, f)

to_json(answer, 'answer_2.json')

print("\n模型训练完成，最优组合已找到。")
print(f"最优激活函数: {a1[0]}")
print(f"最优优化器: {a1[1]}")
print(f"结果已保存至 answer_2.json 文件。")


--- Best Model Found ---
Activation: 'tanh'
Optimizer: 'adam'
Validation Accuracy: 95.93%

模型训练完成，最优组合已找到。
最优激活函数: tanh
最优优化器: adam
结果已保存至 answer_2.json 文件。
